# Qwen3-TTS fine-tuning (VoiceStudio)

Trains Qwen3-TTS on the dataset folder VoiceStudio exported, using Qwen's official `finetuning/` scripts. Everything reusable is cached on your Google Drive, and every cell below calls one script (`voicestudio_qwen3_colab.py`, shipped with the export). **Nothing here has been run on Colab by its author: treat the first run as the test.**

**Checklist**
1. **Runtime > Change runtime type > pick the GPU.** A100, H100 and L4 use the original script (FlashAttention 2). The **G4** (RTX PRO 6000 Blackwell) uses a patched SDPA copy by default (unverified). A **T4** cannot run FlashAttention 2 and needs `ATTN_FALLBACK = True` (unverified, may run out of memory).
2. **Compute units.** A runtime uses units while connected, idle or not, and each GPU type has its own rate (see Colab's runtime menu). Run once with `SMOKE_RUN = True` for a short timing run that projects time and units, then set it back to `False`.
3. Copy the export zip (or folder) to Drive and set `DATA_SOURCE` below to it.
4. Run the cells top to bottom. The first run on a new runtime type installs and may build FlashAttention; later runs reuse the Drive cache.
5. When finished, **disconnect and delete the runtime** (last cell, or Runtime > Disconnect and delete runtime).

In [ ]:
# GPU check: which GPU did Colab give you? (details and the attention path are printed again by the setup cell)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## Parameters
Edit the values in the next cell. `DATA_SOURCE` is where the training data is: the export zip or folder on Drive (or an already unzipped folder under `/content`). The cells after it do the work and hold no settings of their own.

In [ ]:
# ---- Edit these, then run the cells below in order. Re-run this cell and the 'mount Drive' cell after changing them. ----
DRIVE_ROOT = "/content/drive/MyDrive/VoiceStudio-training"  # cache, checkpoints and logs live here
DATA_SOURCE = "/content/drive/MyDrive/VoiceStudio-training/datasets/my-dataset.zip"  # where the training data is: a .zip or a folder on Drive, or an unzipped folder under /content
SPEAKER_NAME = "MySpeaker"  # the name the fine-tuned model answers to
EPOCHS = 3
LR = 2e-05
BATCH_SIZE = 2  # a number, or 'auto' for the suggestion for this GPU (untested estimates, see the GPU section of the output)
BASE_MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
RUN_ID = ''  # empty: a new timestamped run folder every time, so a second run never overwrites the first
KEEP_ALL = True  # True keeps every checkpoint (each is several GB). Nothing is ever deleted while this is True
KEEP_LAST_N = 0  # only with KEEP_ALL = False: keep the newest N epochs (plus KEEP_FIRST / KEEP_FINAL)
KEEP_FIRST = False
KEEP_FINAL = True
ATTN_FALLBACK = 'auto'  # 'auto': FlashAttention 2 on A100/H100/L4, patched SDPA copy on Blackwell/G4; True: force SDPA (unverified); False: force FlashAttention 2
RESTART = False  # True lets a RUN_ID you re-use replace its own checkpoints (the official script cannot resume)
SAMPLE_TEXT = "The lighthouse keeper counted seven ships before the fog rolled in."  # sentence for final.wav after training
SAMPLE_SEED = 1234
CACHE_MODELS_ON_DRIVE = True  # False: download models straight to /content (nothing cached on Drive)
SMOKE_RUN = False  # True: a short timing run (one epoch on the first SMOKE_CLIPS clips, scratch folder) instead of real training
SMOKE_CLIPS = 40
UNITS_PER_HOUR = None  # this runtime's compute units per hour; empty = looked up for known GPUs (owner's figures, unverified)
UNITS_AVAILABLE = 100
COMPARE_EPOCH_SECONDS = None  # optional: seconds per epoch measured on a different GPU, to compare cost per epoch
COMPARE_UNITS_PER_HOUR = None  # that other GPU's rate
TOKENIZER_MODEL = "Qwen/Qwen3-TTS-Tokenizer-12Hz"

## 1. Mount Drive and load the script

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Load the one script that does the work: from the export (DATA_SOURCE) if it is there, else the copy cached on Drive.
import os, sys, zipfile
SCRIPT = 'voicestudio_qwen3_colab.py'
CACHE = os.path.join(DRIVE_ROOT, 'cache')
os.makedirs(CACHE, exist_ok=True)
cached = os.path.join(CACHE, SCRIPT)
fresh = None
if DATA_SOURCE.lower().endswith('.zip') and os.path.isfile(DATA_SOURCE):
    with zipfile.ZipFile(DATA_SOURCE) as z:
        fresh = z.read(SCRIPT) if SCRIPT in z.namelist() else None
elif os.path.isfile(os.path.join(DATA_SOURCE, SCRIPT)):
    fresh = open(os.path.join(DATA_SOURCE, SCRIPT), 'rb').read()
if fresh is not None and (not os.path.isfile(cached) or open(cached, 'rb').read() != fresh):
    open(cached, 'wb').write(fresh)
if not os.path.isfile(cached):
    raise FileNotFoundError(SCRIPT + ' not found: check DATA_SOURCE (it must point at the export zip or folder, which contains it).')
if CACHE not in sys.path:
    sys.path.insert(0, CACHE)
import importlib, voicestudio_qwen3_colab as vs
importlib.reload(vs)
P = vs.params_from(dict(drive_root=DRIVE_ROOT, data_source=DATA_SOURCE, speaker_name=SPEAKER_NAME, epochs=EPOCHS, lr=LR,
    batch_size=BATCH_SIZE, base_model=BASE_MODEL, tokenizer_model=TOKENIZER_MODEL, run_id=RUN_ID, keep_all=KEEP_ALL, keep_last_n=KEEP_LAST_N,
    keep_first=KEEP_FIRST, keep_final=KEEP_FINAL, sample_text=SAMPLE_TEXT, sample_seed=SAMPLE_SEED, cache_models_on_drive=CACHE_MODELS_ON_DRIVE,
    attn_fallback=ATTN_FALLBACK, restart=RESTART, smoke_run=SMOKE_RUN, smoke_clips=SMOKE_CLIPS, units_per_hour=UNITS_PER_HOUR, units_available=UNITS_AVAILABLE,
    compare_epoch_seconds=COMPARE_EPOCH_SECONDS, compare_units_per_hour=COMPARE_UNITS_PER_HOUR), env={})
print('Runner', vs.RUNNER_VERSION, 'loaded from', cached)

## 2. Set up and verify
Creates the Drive layout, installs what is missing, gets FlashAttention (cached wheel, prebuilt wheel or one build saved to Drive) when the GPU needs it, downloads the models once, and prints a pass/fail table plus the time each step took.

In [ ]:
ok = vs.setup(P)

## 3. Check the data
Copies or unzips the dataset to local disk and verifies every `audio` and `ref_audio` path (exists, 24 kHz mono) before any GPU time is spent.

In [ ]:
folder = vs.data_check(P)

## 4. Train (or run the timing run)
With `SMOKE_RUN = True` this runs one epoch on the first few clips and prints the projected time and compute units; with `False` it trains for real. Checkpoints are written to `/content`, then each finished epoch is copied to `runs/<speaker>/<run_id>/checkpoints` on Drive under a temporary name, verified and renamed, and the output says what is now safe on Drive. Progress lines, a heartbeat when quiet, `metrics.csv` and `status.json` show it is alive. Re-running the cell starts a new run unless you re-use a `RUN_ID`.

In [ ]:
vs.train(P)

## 5. Results
Lists the checkpoints on Drive with their sizes.

In [ ]:
vs.results(P)

## 6. Final sample (optional, unverified)
Training already makes `samples/final.wav` when it finishes. Run this to make it again (for example with a new `SAMPLE_TEXT`) from the final checkpoint on Drive, using the usage in Qwen's finetuning README.

In [ ]:
vs.final_sample(P)

## 7. Disconnect
An idle runtime keeps using compute units. Uncomment and run the line below, or use Runtime > Disconnect and delete runtime. (`runtime.unassign()` was announced by the Colab team in 2022; it is not run here.)

In [ ]:
# from google.colab import runtime; runtime.unassign()